In [4]:


!pip install deepface opencv-python-headless tensorflow matplotlib ipywidgets

print("All libraries have been installed successfully!")

All libraries have been installed successfully!


In [ ]:
import cv2
import numpy as np
from deepface import DeepFace
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import matplotlib.pyplot as plt

def take_photo(filename='face.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');

            const capture = document.createElement('button');
            capture.textContent = 'Take a photo';
            capture.style.fontSize = '20px';
            capture.style.padding = '15px';
            capture.style.margin = '10px';
            capture.style.backgroundColor = '#4CAF50';
            capture.style.color = 'white';
            capture.style.border = 'none';
            capture.style.borderRadius = '5px';
            capture.style.cursor = 'pointer';

            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.width = '100%';
            video.style.maxWidth = '500px';
            video.style.margin = '10px auto';

            div.appendChild(capture);
            div.appendChild(video);
            document.body.appendChild(div);

            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(600, true);

            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);

            stream.getVideoTracks()[0].stop();
            div.remove();

            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')

    display(js)
    data = eval_js('takePhoto({})'.format(quality))

    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

def analyze_emotion(image_path):
    try:
        result = DeepFace.analyze(
            img_path=image_path,
            actions=['emotion'],
            enforce_detection=True
        )

        if isinstance(result, list):
            result = result[0]

        emotions = result['emotion']
        dominant_emotion = result['dominant_emotion']

        return {
            'success': True,
            'dominant': dominant_emotion,
            'all_emotions': emotions,
            'confidence': max(emotions.values())
        }

    except Exception as e:
        return {
            'success': False,
            'error': str(e)
        }

print("=" * 50)
print("🔴 Allow the camera when the browser asks")
print("🟢 Press the green 'Take a photo' button")
print("=" * 50)

photo_path = take_photo()

print("Analyzing the image...")

result = analyze_emotion(photo_path)

if result['success']:
    print(f"\n You look: {result['dominant']}")
    print(f"📊 Confidence: {result['confidence']:.2f}%")

    emotions = result['all_emotions']
    emotions = dict(sorted(emotions.items(), key=lambda x: x[1], reverse=True))

    plt.figure(figsize=(10, 5))
    plt.bar(emotions.keys(), emotions.values())
    plt.title('Emotion Percentages')
    plt.ylabel('Percentage (%)')
    plt.xticks(rotation=45)
    plt.show()

else:
    print(f"❌ Error: {result['error']}")


🔴 Allow the camera when the browser asks
🟢 Press the green 'Take a photo' button


<IPython.core.display.Javascript object>